# Testing Model

En este notebook se realizarán pruebas del modelo de clusterización bioinspirado con diferentes funciones de distancia y métodos de selección de características con el objetivo de encontrar la combinación que maximize el rendimiento del algoritmo usando el coeficiente de Silhouette y el indice de Davies-Bouildin para medir la calidad de la clusterización realizada.

In [125]:
import numpy as np
import pandas as pd
from sklearn.metrics import davies_bouldin_score
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Distancia Euclidiana


In [126]:
# Algoritmo bioinspirado en la expansión del reino fungi
class FungiInspiredClustering:
    """
    Algoritmo de clustering bioinspirado en la expansión del micelio del reino fungi.

    El algoritmo simula la expansión caótica local y el reajuste global observado en el crecimiento
    de los hongos para agrupar datos. Utiliza una combinación de búsqueda local con ruido aleatorio
    (expansión caótica) y ajuste global (expansión dirigida) para encontrar grupos de datos de manera eficiente.

    Parámetros
    ----------
    Distance_Function : callable
        Función que calcula la distancia entre dos puntos.
    n_clusters : int, opcional (por defecto=5)
        Número de clusters que se desean formar.
    max_iter : int, opcional (por defecto=100)
        Número máximo de iteraciones que ejecutará el algoritmo.
    local_radius : float, opcional (por defecto=1.0)
        Radio de expansión local que simula ruido aleatorio en la asignación de clusters.
    global_radius : float, opcional (por defecto=5.0)
        Radio de ajuste global para el reajuste de los centroides.
    tolerance : float, opcional (por defecto=1e-4)
        Criterio de convergencia. Si los centroides cambian menos que este valor, el algoritmo se detendrá.

    Atributos
    ---------
    centroids : ndarray de forma (n_clusters, n_features)
        Posiciones finales de los centroides después del ajuste.
    labels : ndarray de forma (n_samples,)
        Etiquetas de cluster para cada punto de los datos.
    """

    def __init__(self,Distance_Function, n_clusters=5, max_iter=100, local_radius=1.0, global_radius=5.0, tolerance=1e-4):
        """
        Inicializa el algoritmo de clustering inspirado en el reino fungi.

        Parámetros
        ----------
        Distance_Function : callable
            Función que calcula la distancia entre dos puntos.
        n_clusters : int, opcional (por defecto=5)
            Número de clusters que se desean formar.
        max_iter : int, opcional (por defecto=100)
            Número máximo de iteraciones que ejecutará el algoritmo.
        local_radius : float, opcional (por defecto=1.0)
            Radio de expansión local que simula ruido aleatorio.
        global_radius : float, opcional (por defecto=5.0)
            Radio de ajuste global para el reajuste de los centroides.
        tolerance : float, opcional (por defecto=1e-4)
            Criterio de convergencia.
        """
        self.Distance_Function = Distance_Function
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.local_radius = local_radius
        self.global_radius = global_radius
        self.tolerance = tolerance

    def fit(self, X):
        """
        Ajusta el modelo FungiInspiredClustering a los datos.

        Parámetros
        ----------
        X : ndarray de forma (n_samples, n_features)
            Conjunto de datos a agrupar.

        Devuelve
        --------
        None
        """
        # Inicializar las "esporas" (centroides iniciales) aleatoriamente
        np.random.seed(42)  # Para reproducibilidad
        random_indices = np.random.choice(X.shape[0], self.n_clusters, replace=False)
        self.centroids = X[random_indices]

        for i in range(self.max_iter):
            # Fase 1: Expansión local (caótica)
            self.labels = self.assign_local_clusters(X)

            # Fase 2: Reajuste global (dirigida)
            old_centroids = self.centroids.copy()
            self.centroids = self.calculate_new_centroids(X)

            # Verificar convergencia
            if self.has_converged(old_centroids):
                print(f"Convergencia alcanzada en la iteración {i+1}")
                break

    def assign_local_clusters(self, X):
        """
        Asigna cada punto de datos al cluster más cercano utilizando la expansión local con ruido.

        Parámetros
        ----------
        X : ndarray de forma (n_samples, n_features)
            Conjunto de datos a agrupar.

        Devuelve
        --------
        labels : ndarray de forma (n_samples,)
            Etiquetas de cluster para cada punto de datos.
        """
        labels = []
        for point in X:
            distances = [self.Distance_Function(point, centroid) for centroid in self.centroids]

            # Expansión local caótica: Agregar ruido a la distancia para simular crecimiento aleatorio
            noisy_distances = [dist + np.random.uniform(-self.local_radius, self.local_radius) for dist in distances]
            labels.append(np.argmin(noisy_distances))
        return np.array(labels)

    def calculate_new_centroids(self, X):
        """
        Recalcula los centroides promediando los puntos en cada cluster y agregando ajuste global.

        Parámetros
        ----------
        X : ndarray de forma (n_samples, n_features)
            Conjunto de datos para el cálculo de nuevos centroides.

        Devuelve
        --------
        centroids : ndarray de forma (n_clusters, n_features)
            Nuevas posiciones de los centroides.
        """
        centroids = []
        for i in range(self.n_clusters):
            points_in_cluster = X[self.labels == i]
            if len(points_in_cluster) > 0:
                # Expansión global: Ajuste de los centroides hacia áreas ricas en puntos
                new_centroid = points_in_cluster.mean(axis=0) + np.random.uniform(-self.global_radius, self.global_radius, size=points_in_cluster.shape[1])
            else:
                # Si el clúster está vacío, reasignar un centroide aleatorio
                new_centroid = X[np.random.choice(X.shape[0])]
            centroids.append(new_centroid)
        return np.array(centroids)

    def has_converged(self, old_centroids):
        """
        Verifica si los centroides han convergido basado en el umbral de tolerancia.

        Parámetros
        ----------
        old_centroids : ndarray de forma (n_clusters, n_features)
            Posiciones anteriores de los centroides.

        Devuelve
        --------
        bool
            True si los centroides han cambiado menos que el valor de tolerancia, de lo contrario False.
        """
        distances = [self.Distance_Function(old, new) for old, new in zip(old_centroids, self.centroids)]
        return np.max(distances) < self.tolerance


In [145]:
# Cargar el dataset
file_path = 'CleanDataset.csv'
df = pd.read_csv(file_path)

In [128]:
# Función para calcular la distancia euclidiana entre dos puntos
def euclidean_distance(point1, point2):
    """
    Calcula la distancia euclidiana entre dos puntos.

    La distancia euclidiana es la longitud del segmento de línea recta entre dos puntos en un espacio euclidiano.

    Parámetros
    ----------
    point1 : ndarray
        El primer punto como un array NumPy.
    point2 : ndarray
        El segundo punto como un array NumPy.

    Devuelve
    --------
    float
        La distancia euclidiana entre point1 y point2.
    """
    return np.sqrt(np.sum((point1 - point2) ** 2))

def minkowski_distance(point1, point2, p=3):
    """
    Calcula la distancia de Minkowski entre dos puntos.

    La distancia de Minkowski es una generalización de las distancias Euclidiana y Manhattan.
    Dependiendo del valor de p, puede representar diferentes métricas de distancia:
    - p = 1: Distancia Manhattan
    - p = 2: Distancia Euclidiana
    - p > 2: Otras variantes de la distancia

    Parámetros
    ----------
    point1 : ndarray
        El primer punto como un array NumPy.
    point2 : ndarray
        El segundo punto como un array NumPy.
    p : int, opcional (por defecto=3)
        El parámetro de orden p que define la métrica de Minkowski. p=3 es el valor por defecto.

    Devuelve
    --------
    float
        La distancia de Minkowski entre point1 y point2.
    """
    return np.sum(np.abs(point1 - point2) ** p) ** (1 / p)

# Función para la distancia Manhattan
def manhattan_distance(point1, point2):
    """
    Calcula la distancia Manhattan entre dos puntos.

    La distancia Manhattan (también conocida como distancia L1) es la suma de las distancias absolutas
    a lo largo de cada dimensión entre dos puntos.

    Parámetros
    ----------
    point1 : ndarray
        El primer punto como un array NumPy.
    point2 : ndarray
        El segundo punto como un array NumPy.

    Devuelve
    --------
    float
        La distancia Manhattan entre point1 y point2.
    """
    return np.sum(np.abs(point1 - point2))


# Pruebas Con Todas Las Columnas

In [129]:
distances=[euclidean_distance, minkowski_distance, manhattan_distance]
for distance in distances:
    fungi_clustering = FungiInspiredClustering(distance, n_clusters=5, max_iter=100, local_radius=0.5,global_radius=2.0)
    fungi_clustering.fit(df.to_numpy())
    # Asignar las etiquetas de clúster al dataframe
    df['Cluster'] = fungi_clustering.labels
    # Tamaño de cada clúster
    cluster_sizes = df['Cluster'].value_counts().sort_index()
    print(f"Tamaño de cada clúster:\n{cluster_sizes}\n")
    # Métricas de rendimiento
    db_score = davies_bouldin_score(df.drop(columns=['Cluster']), df['Cluster'])
    print(f"Davies-Bouldin Score: {db_score}")
    silhouette_avg = silhouette_score(df.to_numpy(), df['Cluster'])
    print(f"Coeficiente de Silhouette: {silhouette_avg} para distancia: {distance}")
    df=df.drop(['Cluster'], axis=1)

Tamaño de cada clúster:
Cluster
0     623
1     664
2    5139
3    2426
4      98
Name: count, dtype: int64

Davies-Bouldin Score: 1.4435858971479152
Coeficiente de Silhouette: 0.34438503682584803 para distancia: <function euclidean_distance at 0x000002A9E5FDB240>
Tamaño de cada clúster:
Cluster
0    1480
1     821
2    4832
3    1587
4     230
Name: count, dtype: int64

Davies-Bouldin Score: 1.6467227134662878
Coeficiente de Silhouette: 0.35245096220679367 para distancia: <function minkowski_distance at 0x000002A9E5FDBA60>
Tamaño de cada clúster:
Cluster
0     109
1     973
2    7271
3     572
4      25
Name: count, dtype: int64

Davies-Bouldin Score: 1.1854566291809645
Coeficiente de Silhouette: 0.47913616450214536 para distancia: <function manhattan_distance at 0x000002A9E5FDB6A0>


# Eliminando características según la matriz de correlación

In [146]:
df=df.drop(columns=['MINIMUM_PAYMENTS','CASH_ADVANCE_TRX','PURCHASES_INSTALLMENTS_FREQUENCY',
                    'TENURE','PRC_FULL_PAYMENT','PURCHASES'])
df

,BALANCE,BALANCE_FREQUENCY,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,CASH_ADVANCE_FREQUENCY,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS
0,40.900749,0.818182,0.00,95.40,0.000000,0.166667,0.000000,0.000000,2.0,1000.0,201.802084
1,3202.467416,0.909091,0.00,0.00,6442.945483,0.000000,0.000000,0.250000,0.0,7000.0,4103.032597
2,2495.148862,1.000000,773.17,0.00,0.000000,1.000000,1.000000,0.000000,12.0,7500.0,622.066742
3,1666.670542,0.636364,1499.00,0.00,205.788017,0.083333,0.083333,0.083333,1.0,7500.0,0.000000
4,817.714335,1.000000,16.00,0.00,0.000000,0.083333,0.083333,0.000000,1.0,1200.0,678.334763
...,...,...,...,...,...,...,...,...,...,...,...
8945,28.493517,1.000000,0.00,291.12,0.000000,1.000000,0.000000,0.000000,6.0,1000.0,325.594462
8946,19.183215,1.000000,0.00,300.00,0.000000,1.000000,0.000000,0.000000,6.0,1000.0,275.861322
8947,23.398673,0.833333,0.00,144.40,0.000000,0.833333,0.000000,0.000000,5.0,1000.0,81.270775
8948,13.457564,0.833333,0.00,0.00,36.558778,0.000000,0.000000,0.166667,0.0,500.0,52.549959


In [131]:
for distance in distances:
    fungi_clustering = FungiInspiredClustering(distance, n_clusters=5, max_iter=100, local_radius=0.5,global_radius=2.0)
    fungi_clustering.fit(df.to_numpy())
    # Asignar las etiquetas de clúster al dataframe
    df['Cluster'] = fungi_clustering.labels
    # Tamaño de cada clúster
    cluster_sizes = df['Cluster'].value_counts().sort_index()
    print(f"Tamaño de cada clúster:\n{cluster_sizes}\n")
    # Métricas de rendimiento
    db_score = davies_bouldin_score(df.drop(columns=['Cluster']), df['Cluster'])
    print(f"Davies-Bouldin Score: {db_score}")
    silhouette_avg = silhouette_score(df.to_numpy(), df['Cluster'])
    print(f"Coeficiente de Silhouette: {silhouette_avg} para distancia: {distance}")
    df=df.drop(['Cluster'], axis=1)

Tamaño de cada clúster:
Cluster
0     551
1     732
2    5291
3    2285
4      91
Name: count, dtype: int64

Davies-Bouldin Score: 1.3031596416034859
Coeficiente de Silhouette: 0.39422812718045996 para distancia: <function euclidean_distance at 0x000002A9E5FDB240>
Tamaño de cada clúster:
Cluster
0    1005
1     848
2    4145
3    2773
4     179
Name: count, dtype: int64

Davies-Bouldin Score: 1.365922531142116
Coeficiente de Silhouette: 0.3083493487301708 para distancia: <function minkowski_distance at 0x000002A9E5FDBA60>
Tamaño de cada clúster:
Cluster
0     125
1    1055
2    6757
3     988
4      25
Name: count, dtype: int64

Davies-Bouldin Score: 1.284384720164371
Coeficiente de Silhouette: 0.4866890381790486 para distancia: <function manhattan_distance at 0x000002A9E5FDB6A0>


# Selección de Características Usando una PCA

In [147]:
df=pd.read_csv('CleanDataset.csv')

In [150]:
pca = PCA(n_components=6)
dfpca=pd.DataFrame(pca.fit_transform(df))
for distance in distances:
    fungi_clustering = FungiInspiredClustering(distance, n_clusters=5, max_iter=100, local_radius=0.5,
                                               global_radius=2.0)
    fungi_clustering.fit(dfpca.to_numpy())
    # Asignar las etiquetas de clúster al dataframe
    dfpca['Cluster'] = fungi_clustering.labels
    # Tamaño de cada clúster
    cluster_sizes = dfpca['Cluster'].value_counts().sort_index()
    print(f"Tamaño de cada clúster:\n{cluster_sizes}\n")
    # Métricas de rendimiento
    silhouette_avg = silhouette_score(dfpca.to_numpy(), dfpca['Cluster'])
    db_score = davies_bouldin_score(dfpca.drop(columns=['Cluster']), dfpca['Cluster'])
    print(f"Davies-Bouldin Score: {db_score}")
    print(f"Coeficiente de Silhouette: {silhouette_avg} para distancia: {distance}")
    dfpca = dfpca.drop(['Cluster'], axis=1)

Tamaño de cada clúster:
Cluster
0     623
1     664
2    5142
3    2423
4      98
Name: count, dtype: int64

Davies-Bouldin Score: 1.4114500710999807
Coeficiente de Silhouette: 0.35135147681406353 para distancia: <function euclidean_distance at 0x000002A9E5FDB240>
Tamaño de cada clúster:
Cluster
0     441
1    2225
2    5841
3     417
4      26
Name: count, dtype: int64

Davies-Bouldin Score: 1.3676078077324842
Coeficiente de Silhouette: 0.40975371372158914 para distancia: <function minkowski_distance at 0x000002A9E5FDBA60>
Tamaño de cada clúster:
Cluster
0    1788
1     864
2    4493
3    1694
4     111
Name: count, dtype: int64

Davies-Bouldin Score: 1.4630147313891795
Coeficiente de Silhouette: 0.3083817670840169 para distancia: <function manhattan_distance at 0x000002A9E5FDB6A0>


# PRUEBAS CON MEJOR DISTANCIA Y SELECCIÓN DE CARACTERÍSTICAS

In [137]:
df=df.drop(columns=['MINIMUM_PAYMENTS','CASH_ADVANCE_TRX','PURCHASES_INSTALLMENTS_FREQUENCY',
                    'TENURE','PRC_FULL_PAYMENT','PURCHASES'])

In [139]:
Ncolonias=[3,5,7,10]
for N in Ncolonias:
    fungi_clustering = FungiInspiredClustering(manhattan_distance, n_clusters=N, max_iter=100, local_radius=0.5,global_radius=2.0)
    fungi_clustering.fit(df.to_numpy())
    # Asignar las etiquetas de clúster al dataframe
    df['Cluster'] = fungi_clustering.labels
    # Tamaño de cada clúster
    cluster_sizes = df['Cluster'].value_counts().sort_index()
    print(f"Tamaño de cada clúster:\n{cluster_sizes}\n")
    # Métricas de rendimiento
    db_score = davies_bouldin_score(df.drop(columns=['Cluster']), df['Cluster'])
    print(f"Davies-Bouldin Score: {db_score}")
    silhouette_avg = silhouette_score(df.to_numpy(), df['Cluster'])
    print(f"Coeficiente de Silhouette: {silhouette_avg} para distancia: manhattan con {N} colonias")
    df=df.drop(['Cluster'], axis=1)

Tamaño de cada clúster:
Cluster
0      66
1    1280
2    7604
Name: count, dtype: int64

Davies-Bouldin Score: 1.0544794118761194
Coeficiente de Silhouette: 0.5258192298952107 para distancia: manhattan con 3 colonias
Tamaño de cada clúster:
Cluster
0     125
1    1055
2    6757
3     988
4      25
Name: count, dtype: int64

Davies-Bouldin Score: 1.284384720164371
Coeficiente de Silhouette: 0.4866890381790486 para distancia: manhattan con 5 colonias
Tamaño de cada clúster:
Cluster
0     333
1    1029
2    1454
3     219
4      25
5    5795
6      95
Name: count, dtype: int64

Davies-Bouldin Score: 1.1761313264756528
Coeficiente de Silhouette: 0.43892868510272026 para distancia: manhattan con 7 colonias
Tamaño de cada clúster:
Cluster
0      37
1     968
2    1412
3     298
4      19
5    5460
6      41
7     228
8     216
9     271
Name: count, dtype: int64

Davies-Bouldin Score: 1.234516103570577
Coeficiente de Silhouette: 0.4201061039874777 para distancia: manhattan con 10 colonias
